<a href="https://colab.research.google.com/github/DilshanBotheju/youtube-comment-corpus-analysis/blob/Task-1/NLP_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CM4608 Natural Language Processing – Individual Coursework
## Public YouTube discussion of Sri Lankan school education reforms

*   **Module:** CM4608
*   **IIT ID:** 20232598
*   **RGU ID:** 2409088

This notebook implements an end-to-end NLP pipeline for video discovery, comment collection and exploratory data analysis

In [1]:
# Install libraries
!pip install -q google-api-python-client langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/CM4608-NLP'
DATA_DIR = os.path.join(BASE_DIR, 'data')
FIG_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Data dir   :', DATA_DIR)
print('Figure dir :', FIG_DIR)

Mounted at /content/drive
Data dir   : /content/drive/MyDrive/CM4608-NLP/data
Figure dir : /content/drive/MyDrive/CM4608-NLP/figures


Package versions and Seeds

In [3]:
import re, json, time, random, hashlib, secrets, platform
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from importlib.metadata import version
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from langdetect import detect, DetectorFactory
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Fixed seed so sampling steps are reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DetectorFactory.seed = SEED

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')

print('Python      :', platform.python_version())
print('pandas      :', pd.__version__)
print('numpy       :', np.__version__)
print('matplotlib  :', matplotlib.__version__)
print('seaborn     :', sns.__version__)
print('api client  :', version('google-api-python-client'))
print('langdetect  :', version('langdetect'))

Python      : 3.13.18
pandas      : 2.2.3
numpy       : 2.4.6
matplotlib  : 3.10.0
seaborn     : 0.13.2
api client  : 2.201.0
langdetect  : 1.0.9


Task 1 – Video discovery and comment collection and EDA on collected data

In [4]:
from datetime import datetime, timezone
from googleapiclient.discovery import build
from google.colab import userdata

# API key is read from Colab Secrets
API_KEY = userdata.get('YT_API_KEY')
youtube = build('youtube', 'v3', developerKey=API_KEY, cache_discovery=False)

# Date of collection, required for reproducibility
COLLECTION_DATE = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
print('Collection date (UTC):', COLLECTION_DATE)

Collection date (UTC): 2026-11-18T06:17:05Z


In [5]:
import json
import time
from collections import Counter
from googleapiclient.errors import HttpError

# Quota cost per request (YouTube Data API v3, default 10,000 units per day)
QUOTA_COST = {'search': 100, 'videos': 1, 'channels': 1, 'commentThreads': 1, 'comments': 1}
quota_used = Counter()

def api_call(request, endpoint, retries=3):
    """Execute a request, count quota, retry server errors, and return (response, error)."""
    for attempt in range(retries):
        try:
            resp = request.execute()
            quota_used[endpoint] += QUOTA_COST[endpoint]
            return resp, None
        except HttpError as e:
            quota_used[endpoint] += QUOTA_COST[endpoint]
            reason = ''
            try:
                reason = json.loads(e.content.decode())['error']['errors'][0]['reason']
            except Exception:
                pass
            # Stop immediately if the daily quota is used up
            if reason in ('quotaExceeded', 'dailyLimitExceeded'):
                raise RuntimeError('YouTube API daily quota exhausted. Re-run tomorrow or reduce queries.') from e
            # Temporary server errors: wait and retry
            if e.resp.status in (500, 503) and attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            # Other errors (e.g. commentsDisabled, videoNotFound) are returned, not raised
            return None, reason or f'http_{e.resp.status}'
    return None, 'max_retries'

def quota_report():
    print(f'Quota used so far: {sum(quota_used.values())} units  {dict(quota_used)}')

Documented search queries (English, Sinhala, Tamil)

In [6]:
import pandas as pd

pd.set_option('display.max_colwidth', 120)

# Every query is anchored to Sri Lanka (country name, Sri Lankan institution or exam)
SEARCH_TERMS = [
    # English
    {'query': 'Sri Lanka education reform 2026', 'lang': 'en'},
    {'query': 'Srilankan education reforms', 'lang': 'en'},
    {'query': 'Sri Lanka new education reforms', 'lang': 'en'},
    {'query': 'Sri Lanka curriculum reform NIE', 'lang': 'en'},
    {'query': 'Sri Lanka school syllabus change', 'lang': 'en'},
    {'query': 'Sri Lanka teachers education reform', 'lang': 'en'},

    # Sinhala
    {'query': 'ශ්‍රී ලංකා අධ්‍යාපන ප්‍රතිසංස්කරණ', 'lang': 'si'},
    {'query': 'නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026', 'lang': 'si'},
    {'query': 'ජාතික අධ්‍යාපන ආයතනය නව විෂය නිර්දේශය', 'lang': 'si'},
    {'query': 'අධ්‍යාපන අමාත්‍යාංශය පාසල් ප්‍රතිසංස්කරණ', 'lang': 'si'},
    # Tamil
    {'query': 'இலங்கை கல்வி சீர்திருத்தம்', 'lang': 'ta'},
    {'query': 'இலங்கை புதிய கல்வி சீர்திருத்தம் 2026', 'lang': 'ta'},
    {'query': 'இலங்கை பாடத்திட்ட மாற்றம்', 'lang': 'ta'},
    {'query': 'இலங்கை பாடசாலை கல்வி மாற்றம்', 'lang': 'ta'},
]

search_terms_df = pd.DataFrame(SEARCH_TERMS)
print(search_terms_df.groupby('lang').size().rename('n_queries'))
search_terms_df

lang
en    6
si    4
ta    4
Name: n_queries, dtype: int64


,query,lang
0,Sri Lanka education reform 2026,en
1,Srilankan education reforms,en
2,Sri Lanka new education reforms,en
3,Sri Lanka curriculum reform NIE,en
4,Sri Lanka school syllabus change,en
5,Sri Lanka teachers education reform,en
6,ශ්‍රී ලංකා අධ්‍යාපන ප්‍රතිසංස්කරණ,si
7,නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026,si
8,ජාතික අධ්‍යාපන ආයතනය නව විෂය නිර්දේශය,si
9,අධ්‍යාපන අමාත්‍යාංශය පාසල් ප්‍රතිසංස්කරණ,si


Video Exploration with relevant search settings to get videos matching to Srilankan Education Reform

In [7]:
# Search settings
SEARCH_REGION = 'LK'                          # bias results towards Sri Lanka
SEARCH_PUBLISHED_AFTER = '2022-01-01T00:00:00Z'
SEARCH_ORDER = 'relevance'
RESULTS_PER_PAGE = 40                         # Number of results per query
PAGES_PER_QUERY = 1

print(f'Estimated search cost: {len(SEARCH_TERMS) * PAGES_PER_QUERY * QUOTA_COST["search"]} units')

search_log = []      # one row per query/page, saved later for reproducibility
candidates = {}      # video_ids
for q in SEARCH_TERMS:
    page_token = None
    for page in range(PAGES_PER_QUERY):
        req = youtube.search().list(
            part='snippet',
            q=q['query'],
            type='video',
            maxResults=RESULTS_PER_PAGE,
            regionCode=SEARCH_REGION,
            publishedAfter=SEARCH_PUBLISHED_AFTER,
            order=SEARCH_ORDER,
            pageToken=page_token,
        )
        resp, err = api_call(req, 'search')
        if resp is None:
            search_log.append({'query': q['query'], 'lang': q['lang'], 'page': page,
                               'n_results': 0, 'error': err, 'retrieved_at': COLLECTION_DATE})
            break
        items = resp.get('items', [])
        for rank, it in enumerate(items):
            vid = it['id']['videoId']
            # the same video can be returned by several queries; keep it once
            entry = candidates.setdefault(vid, {'video_id': vid, 'matched_queries': [], 'best_rank': rank})
            entry['matched_queries'].append(q['query'])
            entry['best_rank'] = min(entry['best_rank'], rank)
        search_log.append({'query': q['query'], 'lang': q['lang'], 'page': page,
                           'n_results': len(items), 'error': None, 'retrieved_at': COLLECTION_DATE})
        page_token = resp.get('nextPageToken')
        if not page_token:
            break

search_log_df = pd.DataFrame(search_log)
print(f'Unique candidate videos: {len(candidates)}')
quota_report()
search_log_df

Estimated search cost: 1400 units
Unique candidate videos: 346
Quota used so far: 1400 units  {'search': 1400}


,query,lang,page,n_results,error,retrieved_at
0,Sri Lanka education reform 2026,en,0,40,None,2026-11-18T06:17:05Z
1,Srilankan education reforms,en,0,40,None,2026-11-18T06:17:05Z
2,Sri Lanka new education reforms,en,0,40,None,2026-11-18T06:17:05Z
3,Sri Lanka curriculum reform NIE,en,0,40,None,2026-11-18T06:17:05Z
4,Sri Lanka school syllabus change,en,0,40,None,2026-11-18T06:17:05Z
5,Sri Lanka teachers education reform,en,0,40,None,2026-11-18T06:17:05Z
6,ශ්‍රී ලංකා අධ්‍යාපන ප්‍රතිසංස්කරණ,si,0,40,None,2026-11-18T06:17:05Z
7,නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026,si,0,40,None,2026-11-18T06:17:05Z
8,ජාතික අධ්‍යාපන ආයතනය නව විෂය නිර්දේශය,si,0,40,None,2026-11-18T06:17:05Z
9,අධ්‍යාපන අමාත්‍යාංශය පාසල් ප්‍රතිසංස්කරණ,si,0,40,None,2026-11-18T06:17:05Z


Video list with video metadata

In [8]:
def get_video_details(video_ids):
    """Fetch title, channel, publication date and statistics in batches of 30 IDs."""
    rows = []
    for i in range(0, len(video_ids), 30):
        req = youtube.videos().list(part='snippet,statistics,status', id=','.join(video_ids[i:i + 30]))
        resp, err = api_call(req, 'videos')
        if resp is None:
            print('videos.list error:', err)
            continue
        for it in resp.get('items', []):
            sn, st = it['snippet'], it.get('statistics', {})
            rows.append({
                'video_id': it['id'],
                'title': sn.get('title', ''),
                'description': sn.get('description', ''),
                'channel_id': sn['channelId'],
                'channel_title': sn.get('channelTitle', ''),
                'video_published_at': sn.get('publishedAt'),
                'video_language': sn.get('defaultLanguage') or sn.get('defaultAudioLanguage'),
                'view_count': int(st.get('viewCount', 0)),
                'like_count': int(st.get('likeCount', 0)) if 'likeCount' in st else None,
                # commentCount is missing when comments are disabled
                'comment_count': int(st['commentCount']) if 'commentCount' in st else None,
                'privacy_status': it['status'].get('privacyStatus'),
                'made_for_kids': it['status'].get('madeForKids'),
            })
    return pd.DataFrame(rows)

videos_df = get_video_details(list(candidates.keys()))
videos_df['matched_queries'] = videos_df['video_id'].map(lambda v: ' | '.join(candidates[v]['matched_queries']))
videos_df['n_matched_queries'] = videos_df['video_id'].map(lambda v: len(candidates[v]['matched_queries']))
videos_df['comments_enabled'] = videos_df['comment_count'].notna()

print(f'Details retrieved for {len(videos_df)} videos')
print(f'Comments disabled / unavailable: {(~videos_df.comments_enabled).sum()}')
quota_report()

Details retrieved for 346 videos
Comments disabled / unavailable: 9
Quota used so far: 1412 units  {'search': 1400, 'videos': 12}


Analysis on countries of fetched channels

In [9]:
channel_ids = videos_df['channel_id'].unique().tolist()
channel_country = {}
for i in range(0, len(channel_ids), 30):
    resp, err = api_call(youtube.channels().list(part='snippet', id=','.join(channel_ids[i:i + 30])), 'channels')
    for it in (resp or {}).get('items', []):
        channel_country[it['id']] = it['snippet'].get('country')
videos_df['channel_country'] = videos_df['channel_id'].map(channel_country)

print(videos_df.drop_duplicates('channel_id')['channel_country'].fillna('not set')
      .value_counts().head(10).rename('n_channels'))
quota_report()

channel_country
LK         105
not set     61
IN          16
US          14
GB           6
CA           4
AU           1
PL           1
AE           1
BD           1
Name: n_channels, dtype: int64
Quota used so far: 1420 units  {'search': 1400, 'videos': 12, 'channels': 8}


Checking the relevance of fetched channels and videos to Sri Lanka and its context

In [10]:
import re

# Patterns to check that a video is about Sri Lanka
SRI_LANKA_PATTERNS = [
    r'sri\s?lanka', r'\blanka', r'ceylon', r'colombo', r'kandy', r'galle', r'jaffna', r'kurunegala',
    r'\bnie\b', r'national institute of education', r'\bg\.?c\.?e\b'
    'ශ්‍රී ලංකා', 'ලංකා', 'ලංකාවේ',
    'இலங்கை', 'ஸ்ரீலங்கா', 'யாழ்ப்பாண', 'மட்டக்களப்பு', 'திருகோணமலை', 'கொழும்பு',
]
# Patterns to check of another country, mainly to drop Indian / Tamil Nadu education videos
FOREIGN_PATTERNS = [
    r'\bindia', r'tamil\s?nadu', r'kerala', r'\bcbse\b', r'\bicse\b', r'\bneet\b', r'\bjee\b',
    r'\bncert\b', r'\bupsc\b', r'\btnpsc\b', r'pakistan', r'bangladesh', r'nepal', r'maldives',
    'தமிழ்நாடு', 'தமிழகம்', 'இந்தியா', 'இந்திய', 'சென்னை',
]
SINHALA_SCRIPT_RE = re.compile(r'[\u0D80-\u0DFF]')

def count_hits(patterns, text):
    """Number of distinct patterns found in the text."""
    return sum(1 for p in patterns if re.search(p, text, flags=re.IGNORECASE))

def sri_lanka_scores(row):
    text = f"{row['title']} {row['description'][:1500]}"
    sl = count_hits(SRI_LANKA_PATTERNS, text)
    sl += int(bool(SINHALA_SCRIPT_RE.search(row['title'])))   # Sinhala is used almost only in Sri Lanka
    sl += int(row['channel_country'] == 'LK')                 # channel registered in Sri Lanka
    return pd.Series({'sri_lanka_score': sl, 'foreign_score': count_hits(FOREIGN_PATTERNS, text)})

videos_df[['sri_lanka_score', 'foreign_score']] = videos_df.apply(sri_lanka_scores, axis=1)
# keep only videos whose Sri Lanka evidence outweighs any other-country evidence
videos_df['is_sri_lanka'] = (videos_df['sri_lanka_score'] > 0) & (videos_df['foreign_score'] < videos_df['sri_lanka_score'])

print(f"Videos with Sri Lanka context: {videos_df['is_sri_lanka'].sum()} / {len(videos_df)}")
videos_df[['sri_lanka_score', 'foreign_score']].describe().round(2)

Videos with Sri Lanka context: 310 / 346


,sri_lanka_score,foreign_score
count,346.00,346.00
mean,2.45,0.08
std,1.60,0.36
min,0.00,0.00
25%,1.00,0.00
50%,2.00,0.00
75%,3.00,0.00
max,7.00,3.00


Relevance to Srilankan Education System

In [11]:
# Topic keywords in English, Sinhala and Tamil
EDUCATION_PATTERNS = [
    r'educat', r'reform', r'curricul', r'syllab', r'school', r'exam', r'\bo/?l\b', r'\ba/?l\b',
    r'grade\s?\d', r'teacher', r'student', r'module', r'vocational', r'classroom',
    'අධ්‍යාපන', 'ප්‍රතිසංස්කරණ', 'පාසල', 'විෂය', 'ශ්‍රේණි', 'ගුරු', 'සිසු', 'විභාග', 'මොඩියුල', 'පෙළ',
    'கல்வி', 'சீர்திருத்த', 'பாடசாலை', 'பாடத்திட்ட', 'ஆசிரிய', 'மாணவ', 'பரீட்சை', 'சாதாரண தர', 'உயர் தர',
]

# Relevance score = number of distinct education keywords in title + description
videos_df['relevance_score'] = videos_df.apply(
    lambda r: count_hits(EDUCATION_PATTERNS, f"{r['title']} {r['description'][:1500]}"), axis=1)

videos_df['relevance_score'].value_counts().sort_index().rename('n_videos').to_frame()

,n_videos
relevance_score,
0,23
1,29
2,79
3,41
4,41
5,27
6,24
7,14
8,19


Video selection approach (comments enabled, more than 20 comments, at least 3 channels)

In [12]:
# Selection settings
MIN_COMMENT_COUNT = 20        # keep videos with more than this many comments
MIN_RELEVANCE_SCORE = 2       # at least two education keywords in title or description
MAX_VIDEOS = 60               # safety limit for API quota

def exclusion_reason(row):
    """Return why a video is excluded, or None if it is eligible. Rules are checked in order."""
    if row['privacy_status'] != 'public':
        return 'not_public'
    if not row['comments_enabled']:
        return 'comments_disabled'
    if not row['is_sri_lanka']:
        return 'not_sri_lanka'
    if row['comment_count'] <= MIN_COMMENT_COUNT:
        return f'comments_<=_{MIN_COMMENT_COUNT}'
    if row['relevance_score'] < MIN_RELEVANCE_SCORE:
        return 'low_relevance'
    return None

videos_df['exclusion_reason'] = videos_df.apply(exclusion_reason, axis=1)
videos_df['exclusion_reason'].fillna('eligible').value_counts().rename('n_videos').to_frame()

,n_videos
exclusion_reason,
comments_<=_20,260
eligible,35
not_sri_lanka,35
comments_disabled,9
low_relevance,7


Channel and Video Diversity

In [13]:
eligible = videos_df[videos_df.exclusion_reason.isna()].copy()

# Round-robin across channels: every channel's top video first, then second videos, and so on.
# This stops one large channel from filling the whole selection.
eligible = eligible.sort_values('comment_count', ascending=False)
eligible['rank_in_channel'] = eligible.groupby('channel_id').cumcount()
selected_videos = (eligible.sort_values(['rank_in_channel', 'comment_count'], ascending=[True, False])
                           .head(MAX_VIDEOS)
                           .reset_index(drop=True))

print(f'Eligible videos   : {len(eligible)}')
print(f'Selected videos   : {len(selected_videos)}')
print(f'Distinct channels : {selected_videos.channel_id.nunique()}')
print(f'Expected comments (statistics.commentCount): {selected_videos.comment_count.sum()}')

# Brief requirement: videos from at least three distinct channels
assert selected_videos.channel_id.nunique() >= 3, 'Fewer than 3 channels - add search terms or relax filters'
selected_videos[['video_id', 'channel_title', 'channel_country', 'title', 'comment_count',
                 'sri_lanka_score', 'relevance_score']]

Eligible videos   : 35
Selected videos   : 35
Distinct channels : 29
Expected comments (statistics.commentCount): 5400.0


,video_id,channel_title,channel_country,title,comment_count,sri_lanka_score,relevance_score
0,-pHFRz2p-Nw,Maths with Chathura Madushan,US,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න,730.0,1,8
1,M8-A7KctUdE,Paryeshakaya,LK,මේ සිසුවාගේ අධ්‍යාපන ගමන ඔබව පුදුමයට පත් කරයි,652.0,2,6
2,R0S8yJajtuI,Ada Derana,LK,2026 නව අධ්‍යාපන ප්‍රතිසංස්කරණ - පාසල් පැවැත්වෙන කාලසීමාවේත් වෙනසක්,589.0,5,3
3,Srr-sYiIDuI,BBC News සිංහල,LK,නව අධ්‍යාපන ප්‍රතිසංස්කරණ සම්බන්ධයෙන් මාධ්‍යවේදීන් නැගූ ප්‍රශ්නවලට ලැබුණු පිළිතුරු | BBC News සිංහල,512.0,2,5
4,FKoS8yJOKWo,Edu Info,LK,"R1 | Grade 6 Subject Selection Guide for Teachers, Students & Parents | Education Reform 2026",264.0,1,9
5,ZYeuMI3SQzw,GK Challenge,None,ගුරු විභාගය සඳහා වැදගත් නව අධ්‍යාපන ප්‍රතිසංස්කරණ|සාමාන්‍ය දැනීම #gk#guru#ගුරු#teacher#දැනුම#2026,206.0,5,10
6,ARUFRrUAfi4,KK Academy SL,LK,උසස් ජාතික ඩිප්ලොමා : සම්පූර්ණයෙන්ම නොමිලේ : SLIATE New Intake 2025 | All HND Courses & How to Apply,173.0,4,4
7,6gF52_MgDiE,Wanithi Bookshop,LK,2026 ශ්‍රී ලංකාවේ වෙනස්වන අධ්‍යාපන ක්‍රමය සහ ගුරුවරයා 🎓 Sri Lankan Educational System | Dr. Harini,162.0,6,10
8,D02nj8GVQYA,Firstpost,IN,Gay Website in School Syllabus Forces Sri Lanka to Delay Education Reforms | Firstpost Live | N18G,152.0,2,5
9,wyzYk_NyOCQ,VFM RADIO 107,US,2026 පාසල් වල උගන්වන්නේ මෙහෙමයි.. අධ්‍යාපන අමාත්‍යංශයෙ නිල වීඩියෝව මෙන්න..,147.0,7,2


Manual Review on Suggested Videos

In [14]:
# After checking the selected-videos table above, list any video IDs to remove, with a reason
EXCLUDE_VIDEO_IDS = {
    # 'VIDEO_ID': 'off-topic: university fees, not school education',
    'M8-A7KctUdE':'off-topic',
    'ARUFRrUAfi4':'off-topic',
    'RCEV4TgKXk8':'off-topic',
    'IEn1ESZlMlc':'off-topic',
    'g1YMZC9ce5I':'off-topic',
    'QcYm2EKhrC4':'off-topic'
}

# Warn about typos: IDs that are not among the candidate videos
unknown = set(EXCLUDE_VIDEO_IDS) - set(videos_df['video_id'])
if unknown:
    print('Not found (check spelling):', sorted(unknown))

# Mark removed videos (only those that passed the automatic rules, so earlier reasons are kept)
manual_mask = videos_df['video_id'].isin(EXCLUDE_VIDEO_IDS) & videos_df['exclusion_reason'].isna()
videos_df.loc[manual_mask, 'exclusion_reason'] = 'manual_exclusion'
videos_df['manual_exclusion_note'] = videos_df['video_id'].map(EXCLUDE_VIDEO_IDS)

# Re-run the channel-diverse selection without the removed videos
eligible = videos_df[videos_df.exclusion_reason.isna()].copy()
eligible = eligible.sort_values('comment_count', ascending=False)
eligible['rank_in_channel'] = eligible.groupby('channel_id').cumcount()
selected_videos = (eligible.sort_values(['rank_in_channel', 'comment_count'], ascending=[True, False])
                           .head(MAX_VIDEOS)
                           .reset_index(drop=True))

print(f'Manually removed videos : {manual_mask.sum()}')
print(f'Selected videos         : {len(selected_videos)}')
print(f'Distinct channels       : {selected_videos.channel_id.nunique()}')
assert selected_videos.channel_id.nunique() >= 3, 'Fewer than 3 channels after manual removal'

# Table of what was removed and why
videos_df.loc[manual_mask, ['video_id', 'channel_title', 'title', 'comment_count', 'manual_exclusion_note']]

Manually removed videos : 6
Selected videos         : 29
Distinct channels       : 23


,video_id,channel_title,title,comment_count,manual_exclusion_note
86,RCEV4TgKXk8,ThilaSpace,One day Alevel results were out - al bio to medical daculty - advanced level biology to med student,136.0,off-topic
90,M8-A7KctUdE,Paryeshakaya,මේ සිසුවාගේ අධ්‍යාපන ගමන ඔබව පුදුමයට පත් කරයි,652.0,off-topic
110,IEn1ESZlMlc,Study diary,The Most Satisfying Feeling as a Student ✨📚🖊️ @Studydiary-u5k#study#motivation,130.0,off-topic
120,QcYm2EKhrC4,Ruvini Nanayakkara,London Syllabus vs National Syllabus in Sri Lanka | Cambridge | Edexcel Syllabus | Local Syllabus 🇱🇰,38.0,off-topic
224,g1YMZC9ce5I,Channel NIE,නීතිය හා යුක්තිය 01 (1 පාඩම) - 11 ශ්‍රේණිය (පුරවැසි අධ්‍යාපනය),43.0,off-topic
240,ARUFRrUAfi4,KK Academy SL,උසස් ජාතික ඩිප්ලොමා : සම්පූර්ණයෙන්ම නොමිලේ : SLIATE New Intake 2025 | All HND Courses & How to Apply,173.0,off-topic


Comment parser with pseudonymised authors

In [15]:
import hashlib
import secrets

# Data minimisation: author names are not stored; author channel IDs are salted and hashed.
# The salt is never saved, so hashes cannot be reversed, but repeat posters can still be detected.
SALT = secrets.token_hex(16)

def pseudonymise(value):
    if not value:
        return None
    return hashlib.sha256((SALT + value).encode('utf-8')).hexdigest()[:16]

def parse_comment(c, video_id, is_reply, parent_id, total_reply_count=0):
    """Keep only the comment text and the minimum metadata needed for analysis."""
    sn = c['snippet']
    return {
        'comment_id': c['id'],
        'video_id': video_id,
        'thread_id': parent_id if is_reply else c['id'],   # groups a comment with its replies
        'parent_id': parent_id,
        'is_reply': is_reply,
        'text_raw': sn.get('textOriginal') or sn.get('textDisplay') or '',   # unmodified text
        'author_hash': pseudonymise(sn.get('authorChannelId', {}).get('value')),
        'comment_like_count': sn.get('likeCount', 0),
        'total_reply_count': total_reply_count,
        'comment_published_at': sn.get('publishedAt'),
        'comment_updated_at': sn.get('updatedAt'),
        'collected_at': COLLECTION_DATE,
    }

Comment Collection

In [16]:
# Collection settings
MAX_COMMENTS_PER_VIDEO = 1500
COMMENT_ORDER = 'time'        # newest first; deterministic paging
INCLUDE_REPLIES = True        # replies are kept and flagged with is_reply

def fetch_all_replies(parent_id, video_id):
    """commentThreads returns at most 5 replies inline; fetch the full reply list here."""
    rows, page_token = [], None
    while True:
        req = youtube.comments().list(part='snippet', parentId=parent_id, maxResults=100,
                                      textFormat='plainText', pageToken=page_token)
        resp, err = api_call(req, 'comments')
        if resp is None:
            break
        rows.extend(parse_comment(r, video_id, True, parent_id) for r in resp.get('items', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return rows

def fetch_comments(video_id, max_comments=MAX_COMMENTS_PER_VIDEO):
    """Page through top-level comments (100 per request) and their replies."""
    rows, page_token, status = [], None, 'ok'
    while len(rows) < max_comments:
        req = youtube.commentThreads().list(part='snippet,replies', videoId=video_id, maxResults=100,
                                            order=COMMENT_ORDER, textFormat='plainText',
                                            pageToken=page_token)
        resp, err = api_call(req, 'commentThreads')
        if resp is None:
            status = err          # e.g. commentsDisabled
            break
        for th in resp.get('items', []):
            top = th['snippet']['topLevelComment']
            n_replies = th['snippet'].get('totalReplyCount', 0)
            rows.append(parse_comment(top, video_id, False, None, n_replies))
            if INCLUDE_REPLIES and n_replies > 0:
                inline = th.get('replies', {}).get('comments', [])
                if n_replies > len(inline):        # thread has more replies than returned inline
                    rows.extend(fetch_all_replies(top['id'], video_id))
                else:
                    rows.extend(parse_comment(r, video_id, True, top['id']) for r in inline)
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return rows, status

Run collection

In [17]:
all_rows, video_status = [], {}

for i, v in selected_videos.iterrows():
    rows, status = fetch_comments(v['video_id'])
    all_rows.extend(rows)
    video_status[v['video_id']] = {'status': status, 'n_retrieved': len(rows)}
    # 'reported' is statistics.commentCount; it includes comments the API does not return
    print(f"[{i + 1:>2}/{len(selected_videos)}] {v['video_id']}  {status:<20} "
          f"retrieved={len(rows):>5}  reported={v['comment_count']:>5}  {v['channel_title'][:30]}")

comments_all = pd.DataFrame(all_rows)
print(f'\nTotal rows retrieved: {len(comments_all)}')
quota_report()

[ 1/29] -pHFRz2p-Nw  ok                   retrieved=  730  reported=730.0  Maths with Chathura Madushan 
[ 2/29] R0S8yJajtuI  ok                   retrieved=  589  reported=589.0  Ada Derana
[ 3/29] Srr-sYiIDuI  ok                   retrieved=  516  reported=512.0  BBC News සිංහල
[ 4/29] FKoS8yJOKWo  ok                   retrieved=  264  reported=264.0  Edu Info
[ 5/29] ZYeuMI3SQzw  ok                   retrieved=  206  reported=206.0  GK Challenge
[ 6/29] 6gF52_MgDiE  ok                   retrieved=  162  reported=162.0  Wanithi Bookshop
[ 7/29] D02nj8GVQYA  ok                   retrieved=  152  reported=152.0  Firstpost
[ 8/29] wyzYk_NyOCQ  ok                   retrieved=  147  reported=147.0  VFM RADIO 107
[ 9/29] mTKTus6XaWA  ok                   retrieved=  148  reported=146.0  EXED Vibe
[10/29] 4A3C19_JGHU  ok                   retrieved=  105  reported=105.0  Ashani Science LAB
[11/29] SlRJNlPAj8A  ok                   retrieved=  101  reported=101.0  Manoj Dhananjaya
[12/29] TZ

In [18]:
# The same comment ID should never appear twice; remove any API-level duplicates
n_before = len(comments_all)
comments_all = comments_all.drop_duplicates(subset='comment_id', keep='first').reset_index(drop=True)
n_dup_ids = n_before - len(comments_all)

# Videos whose comments could not be retrieved
failed = {k: v for k, v in video_status.items() if v['status'] != 'ok' or v['n_retrieved'] == 0}
print(f'Duplicate comment IDs removed: {n_dup_ids}')
print(f'Videos that failed or returned no comments: {len(failed)}')
for k, v in failed.items():
    print('  ', k, v)

Duplicate comment IDs removed: 6
Videos that failed or returned no comments: 0


In [28]:
# Corpus balance settings
MAX_VIDEO_SHARE = 0.30        # brief: no single video above 30% of the corpus
TARGET_MIN_COMMENTS = 3000    # brief: at least 3,000 unique comments

def apply_video_cap(df, max_share, seed):
    """Down-sample whole threads of any video above max_share of the corpus.
    Whole threads are kept or dropped so replies never lose their parent comment.
    Repeats until no video exceeds the limit."""
    df = df.copy()
    capped = {}
    while True:
        counts = df['video_id'].value_counts()
        total = len(df)
        top_vid, top_n = counts.index[0], counts.iloc[0]
        if top_n / total <= max_share:
            return df, capped
        # largest size this video can have so that its share is max_share
        others = total - top_n
        keep_n = int(np.floor(max_share * others / (1 - max_share)))
        sub = df[df['video_id'] == top_vid]
        thread_sizes = sub.groupby('thread_id').size().sample(frac=1, random_state=seed)
        keep_threads = thread_sizes[thread_sizes.cumsum() <= keep_n].index
        drop_idx = sub[~sub['thread_id'].isin(keep_threads)].index
        df = df.drop(drop_idx)
        capped[top_vid] = {'before': int(top_n), 'after': int(len(sub) - len(drop_idx))}

comments_raw, capped_videos = apply_video_cap(comments_all, MAX_VIDEO_SHARE, SEED)
comments_raw = comments_raw.reset_index(drop=True)

print(f'Videos down-sampled by the {MAX_VIDEO_SHARE:.0%} cap: {capped_videos}')
print(f'Final corpus size: {len(comments_raw)} (target >= {TARGET_MIN_COMMENTS}: '
      f'{len(comments_raw) >= TARGET_MIN_COMMENTS})')
print(f'Largest video share: {comments_raw.video_id.value_counts(normalize=True).iloc[0]:.1%}')

Videos down-sampled by the 30% cap: {}
Final corpus size: 4228 (target >= 3000: True)
Largest video share: 17.3%


Save Files

In [29]:
# Attach video and channel metadata to each comment
video_meta_cols = ['video_id', 'channel_id', 'channel_title', 'channel_country', 'title', 'video_published_at',
                   'video_language', 'view_count', 'comment_count', 'matched_queries']
comments_raw = comments_raw.merge(selected_videos[video_meta_cols].rename(
    columns={'title': 'video_title', 'comment_count': 'video_reported_comment_count'}), on='video_id', how='left')

# comments_raw.csv is the unmodified raw-data file used by Task 2; utf-8-sig keeps Sinhala/Tamil readable in Excel
comments_all.to_csv(os.path.join(DATA_DIR, 'comments_raw_uncapped.csv'), index=False, encoding='utf-8-sig')
comments_raw.to_csv(os.path.join(DATA_DIR, 'comments_raw.csv'), index=False, encoding='utf-8-sig')
videos_df.drop(columns=['description']).to_csv(os.path.join(DATA_DIR, 'videos_candidates.csv'),
                                               index=False, encoding='utf-8-sig')
selected_videos.drop(columns=['description']).to_csv(os.path.join(DATA_DIR, 'videos_selected.csv'),
                                                     index=False, encoding='utf-8-sig')
search_log_df.to_csv(os.path.join(DATA_DIR, 'search_log.csv'), index=False, encoding='utf-8-sig')

Collection Log

In [30]:
# Every setting and decision of Tasks in one file, so the collection can be repeated
collection_log = {
    'collection_date_utc': COLLECTION_DATE,
    'api': 'YouTube Data API v3',
    'seed': SEED,
    'search_settings': {'region': SEARCH_REGION, 'published_after': SEARCH_PUBLISHED_AFTER,
                        'order': SEARCH_ORDER, 'results_per_page': RESULTS_PER_PAGE,
                        'pages_per_query': PAGES_PER_QUERY},
    'search_terms': SEARCH_TERMS,
    'sri_lanka_patterns': SRI_LANKA_PATTERNS,
    'foreign_patterns': FOREIGN_PATTERNS,
    'education_patterns': EDUCATION_PATTERNS,
    'manual_exclusions': EXCLUDE_VIDEO_IDS,
    'selection_settings': {'min_comment_count_exclusive': MIN_COMMENT_COUNT,
                           'min_relevance_score': MIN_RELEVANCE_SCORE, 'max_videos': MAX_VIDEOS},
    'collection_settings': {'max_comments_per_video': MAX_COMMENTS_PER_VIDEO,
                            'comment_order': COMMENT_ORDER, 'include_replies': INCLUDE_REPLIES},
    'balance_settings': {'max_video_share': MAX_VIDEO_SHARE, 'target_min_comments': TARGET_MIN_COMMENTS},
    'n_candidate_videos': int(len(videos_df)),
    'exclusion_counts': videos_df['exclusion_reason'].fillna('eligible').value_counts().to_dict(),
    'n_selected_videos': int(len(selected_videos)),
    'n_channels': int(selected_videos.channel_id.nunique()),
    'video_status': video_status,
    'duplicate_ids_removed': int(n_dup_ids),
    'capped_videos': capped_videos,
    'n_comments_uncapped': int(len(comments_all)),
    'n_comments_final': int(len(comments_raw)),
    'quota_used': dict(quota_used),
    'privacy': 'Author names not stored; author channel IDs replaced by salted SHA-256 hash (salt discarded).',
}
with open(os.path.join(DATA_DIR, 'collection_log.json'), 'w', encoding='utf-8') as f:
    json.dump(collection_log, f, ensure_ascii=False, indent=2)

print('Saved files:')
for fn in sorted(os.listdir(DATA_DIR)):
    print(f'  {fn:<32} {os.path.getsize(os.path.join(DATA_DIR, fn)) / 1024:>8.1f} KB')

Saved files:
  collection_log.json                   6.8 KB
  comments_eda_flags.csv              385.7 KB
  comments_raw.csv                   3116.7 KB
  comments_raw_uncapped.csv          1484.2 KB
  eda_summary.json                      3.2 KB
  search_log.csv                        1.3 KB
  videos_candidates.csv               129.7 KB
  videos_selected.csv                  13.3 KB


Summarise the corpus

In [31]:
# Reload from Drive so the EDA runs without calling the API again
df = pd.read_csv(os.path.join(DATA_DIR, 'comments_raw.csv'), encoding='utf-8-sig',
                 keep_default_na=False, na_values=[''])
df['text_raw'] = df['text_raw'].fillna('')
df['comment_published_at'] = pd.to_datetime(df['comment_published_at'], utc=True)
df['video_published_at'] = pd.to_datetime(df['video_published_at'], utc=True)
videos_sel = pd.read_csv(os.path.join(DATA_DIR, 'videos_selected.csv'), encoding='utf-8-sig')

summary = pd.Series({
    'Total comments': len(df),
    'Top-level comments': int((~df.is_reply).sum()),
    'Replies': int(df.is_reply.sum()),
    'Videos': df.video_id.nunique(),
    'Channels': df.channel_id.nunique(),
    'Unique (pseudonymised) authors': df.author_hash.nunique(),
    'Earliest comment': df.comment_published_at.min().date(),
    'Latest comment': df.comment_published_at.max().date(),
    'Earliest video': df.video_published_at.min().date(),
    'Latest video': df.video_published_at.max().date(),
}, name='value')
summary.to_frame()

,value
Total comments,4228
Top-level comments,2598
Replies,1630
Videos,29
Channels,23
Unique (pseudonymised) authors,2622
Earliest comment,2024-03-18
Latest comment,2026-11-15
Earliest video,2024-03-17
Latest video,2026-10-15
